# Focus Area 4 — Agro-Ecological Zoning with PyAEZ v2.2

Bridges climate monitoring (FA1–FA3) into agricultural interpretation using
**PyAEZ v2.2** (FAO / Asian Institute of Technology).

**What it computes:**
- Thermal Climate Classification (Module I)
- Length of Growing Period — LGP (Module I)
- Agro-Ecological Zone spatial maps
- Crop suitability assessment
**Countries:** Kenya ·  Ethiopia · Tanzania · Eritrea · Djibouti · Somalia · Sudan



In [ ]:
# @title ### 0) Configuration {"display-mode":"form"}
# @markdown Set COUNTRY and PRECIP_SOURCE.
# @markdown PRECIP_SOURCE = product that scored highest in FA2 for this country.

import os, numpy as np
from datetime import date

COUNTRY       = "Kenya"  # @param ["Kenya","Ethiopia", "Tanzania", "Eritrea", "Djibouti", "Somalia", "Sudan"]
PRECIP_SOURCE = 'CHIRPS'     # @param['CHIRPS','TAMSAT','ERA5','IMERG']
LT_START      = 2020
LT_END        = 2024

COUNTRY_BBOX = {
    'Kenya':    [33.9,  -4.7,  41.9,   5.1],
    'Ethiopia': [33.0,   3.0,  48.0,  15.0],
    'Tanzania': [29.0, -12.0,  41.0,  -1.0],
    'Eritrea':  [36.5,  12.5,  43.5,  18.0],
    'Djibouti': [41.5,  10.9,  43.5,  12.7],
    'Somalia':  [41.0,  -1.7,  51.5,  12.0],
    'Sudan':    [21.8,   8.7,  38.7,  22.2],
}
xmin, ymin, xmax, ymax = COUNTRY_BBOX[COUNTRY]

LOCAL_CACHE_DIR = f'./Datasets/{COUNTRY}'
RESULTS_DIR     = f'./outputs/FA4_AEZ_{COUNTRY}_{date.today().isoformat()}'
os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR,     exist_ok=True)

print('\u2705 CONFIG set.')
print(f'   Country         : {COUNTRY}')
print(f'   Bbox            : [{xmin}, {ymin}, {xmax}, {ymax}]')
print(f'   Precip source   : {PRECIP_SOURCE}')
print(f'   Long-term period: {LT_START}\u2013{LT_END}')
print(f'   Outputs         : {RESULTS_DIR}')

# ── Overall "Run All" progress tracker ────────────────────────────────────────
# A single shared progress bar that advances once per major Step (0-8, matching
# this notebook's own Step numbering) as it runs top to bottom. Re-displayed at
# the start of each Step's first cell so it stays visible near wherever the
# notebook currently is, instead of only appearing once at the very top.
import time as _nb_time
import ipywidgets as _nb_widgets
from IPython.display import display as _nb_display

NOTEBOOK_STEP_NAMES = {
    0: 'Configuration',
    1: 'GEE Authentication',
    2: 'Install dependencies & mount Shared Drive',
    3: 'Load climate inputs from Shared Drive',
    4: 'Prepare monthly climate arrays',
    5: 'Module I — Climate Regime Analysis',
    6: 'Crop suitability assessment',
    7: 'Results visualisation',
    8: 'Cache results to Shared Drive',
}
_NOTEBOOK_TOTAL_STEPS = max(NOTEBOOK_STEP_NAMES)
_notebook_start_time = _nb_time.time()

_notebook_progress_bar = _nb_widgets.IntProgress(
    min=0, max=_NOTEBOOK_TOTAL_STEPS, value=0,
    description='Overall progress:',
    style={'description_width': 'initial', 'bar_color': '#2e7d32'},
    bar_style='info',
    layout=_nb_widgets.Layout(width='50%'),
)
_notebook_progress_label = _nb_widgets.Label(value=f'Step 0/{_NOTEBOOK_TOTAL_STEPS} — not started yet')

def advance_notebook_progress(step_num):
    """Bump the shared overall-progress bar to `step_num` and re-display it
    here so it's visible at this point in the notebook too."""
    elapsed = _nb_time.time() - _notebook_start_time
    mins, secs = divmod(int(elapsed), 60)
    step_name = NOTEBOOK_STEP_NAMES.get(step_num, '')
    _notebook_progress_bar.value = step_num
    if step_num >= _NOTEBOOK_TOTAL_STEPS:
        _notebook_progress_bar.bar_style = 'success'
    _notebook_progress_label.value = (
        f'Step {step_num}/{_NOTEBOOK_TOTAL_STEPS} — {step_name}  '
        f'(elapsed {mins:02d}:{secs:02d})'
    )
    _nb_display(_nb_widgets.HBox([_notebook_progress_bar, _notebook_progress_label]))

print('✅ Overall run-all progress tracker initialised.')
advance_notebook_progress(0)


In [ ]:
# @title ### 1) GEE Authentication {"display-mode":"form"}

try:
    advance_notebook_progress(1)
except NameError:
    pass  # overall progress tracker not initialised (Config cell not yet run) -- safe to ignore

import ee, json, gdown

SA_KEY_FILE_ID = '181IKB3sJ3iUn6ZOZbg50htgH2JKcxFkT'
SA_KEY_PATH    = 'service_account_key.json'

print('\U0001f511 Downloading service account key...')
gdown.download(
    f'https://drive.google.com/uc?id={SA_KEY_FILE_ID}',
    SA_KEY_PATH, quiet=True)

if not os.path.exists(SA_KEY_PATH) or os.path.getsize(SA_KEY_PATH) == 0:
    raise RuntimeError('\u274c Key file download failed. Share as Anyone with the link.')

with open(SA_KEY_PATH) as f:
    _key = json.load(f)

credentials = ee.ServiceAccountCredentials(
    email=_key['client_email'], key_file=SA_KEY_PATH)
ee.Initialize(credentials)
print(f'\u2705 GEE authenticated as: {_key["client_email"]}')

In [ ]:
# @title ### 2) Install dependencies & mount Shared Drive {"display-mode":"form"}
try:
    advance_notebook_progress(2)
except NameError:
    pass  # overall progress tracker not initialised (Config cell not yet run) -- safe to ignore

print('Installing dependencies...')

import subprocess
!apt-get install -y libgdal-dev gdal-bin > /dev/null 2>&1
_gdal_ver = subprocess.check_output(['gdal-config','--version']).decode().strip()
print(f'   System GDAL: {_gdal_ver}')
!pip install gdal[numpy]=={_gdal_ver} -q
!pip install pyaez==2.2 numba scipy -q
!pip install xarray rioxarray rasterio cartopy -q
print('\u2705 Dependencies installed.')

import os, json, shutil, importlib, inspect
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from datetime import date
from IPython.display import display
from google.colab import drive

# ── PyAEZ v2.2 imports ───────────────────────────────────────────────────────
PYAEZ_AVAILABLE     = False
ClimateRegime       = None
UtilityCalculations = None

try:
    from pyaez import ClimateRegime as _CR
    ClimateRegime   = _CR
    PYAEZ_AVAILABLE = True
    print('\u2705 PyAEZ ClimateRegime imported.')
    _cr_test    = ClimateRegime.ClimateRegime()
    _cr_methods = [m for m in dir(_cr_test) if not m.startswith('_')]
    print(f'   Methods: {_cr_methods}')
except ImportError as e:
    print(f'\u26a0\ufe0f  ClimateRegime import failed: {e}')

# v2.2 renamed UtilityCalculations -> UtilitiesCalc
for _uc_attr in ['UtilitiesCalc', 'UtilityCalculations']:
    try:
        import pyaez as _p
        _uc = getattr(_p, _uc_attr, None)
        if _uc is not None:
            UtilityCalculations = _uc
            print(f'\u2705 PyAEZ {_uc_attr} imported.')
            break
    except Exception:
        pass

if UtilityCalculations is None:
    try:
        import pyaez as _pyaez_pkg
        _pkg_dir = os.path.dirname(_pyaez_pkg.__file__)
        for _fname in ['UtilitiesCalc.py', 'UtilityCalculations.py']:
            _uc_path = os.path.join(_pkg_dir, _fname)
            if os.path.exists(_uc_path):
                _spec   = importlib.util.spec_from_file_location(_fname[:-3], _uc_path)
                _uc_mod = importlib.util.module_from_spec(_spec)
                _spec.loader.exec_module(_uc_mod)
                UtilityCalculations = _uc_mod
                print(f'\u2705 {_fname} loaded via direct path.')
                break
    except Exception:
        pass

if UtilityCalculations is None:
    print('\u26a0\ufe0f  UtilitiesCalc not found — Ra will use latitude approximation.')
if not PYAEZ_AVAILABLE:
    print('\u26a0\ufe0f  PyAEZ unavailable — manual FAO-56 calculations will be used.')

# ── Shared Drive ─────────────────────────────────────────────────────────────
print('\n\U0001f4c2 Mounting Google Drive...')
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive', force_remount=False)
else:
    print('   Drive already mounted.')

SHARED_DRIVE_ROOT = '/content/drive/Shareddrives/NOAA-workshop2'
DATASETS_ROOT     = os.path.join(SHARED_DRIVE_ROOT, 'Datasets')
COUNTRY_DRIVE_DIR = os.path.join(DATASETS_ROOT, COUNTRY)

if not os.path.exists(SHARED_DRIVE_ROOT):
    print('\u274c Shared Drive not found.')
    DRIVE_CACHE_AVAILABLE = False
else:
    os.makedirs(COUNTRY_DRIVE_DIR, exist_ok=True)
    DRIVE_CACHE_AVAILABLE = True
    print(f'\u2705 Shared Drive ready: {COUNTRY_DRIVE_DIR}')

def try_load_from_drive(local_path, filename):
    if not DRIVE_CACHE_AVAILABLE: return False
    src = os.path.join(COUNTRY_DRIVE_DIR, filename)
    if os.path.exists(src):
        os.makedirs(os.path.dirname(local_path) or '.', exist_ok=True)
        shutil.copy(src, local_path)
        print(f'   \u2705 Loaded from Drive: {filename}')
        return True
    return False

def cache_to_drive(local_path, filename):
    if not DRIVE_CACHE_AVAILABLE: return
    dest = os.path.join(COUNTRY_DRIVE_DIR, filename)
    try:
        shutil.copy(local_path, dest)
        print(f'   \u2601\ufe0f  Cached to Drive: {filename}')
    except Exception as e:
        print(f'   \u26a0\ufe0f  Cache failed: {e}')

In [ ]:
import ee
import os # Ensure os is imported for os.remove

def subset_to_bbox(ds, xmin, xmax, ymin, ymax):
    lat_dim = next((c for c in ds.coords if 'lat' in c.lower() or c == 'y'), None)
    lon_dim = next((c for c in ds.coords if 'lon' in c.lower() or c == 'x'), None)
    if not lat_dim or not lon_dim: return ds
    lv = ds[lat_dim].values
    ls = slice(ymax, ymin) if lv[0] > lv[-1] else slice(ymin, ymax)
    subset = ds.sel({lat_dim: ls, lon_dim: slice(xmin, xmax)})
    # BUGFIX: xarray's .sel() with a slice silently returns an empty array
    # when the bbox doesn't overlap the dataset's actual coordinates -- it
    # doesn't raise. That produced exactly this failure for Tanzania: a
    # 0-row/0-col grid that only blew up several cells later with a
    # confusing "zero-size array to reduction operation" error. Fail here
    # instead, with the actual vs requested ranges, so it's obvious whether
    # this is a stale/mismatched cached file.
    if subset.sizes.get(lat_dim, 0) == 0 or subset.sizes.get(lon_dim, 0) == 0:
        raise ValueError(
            f"subset_to_bbox produced an EMPTY grid for bbox "
            f"[{xmin}, {ymin}, {xmax}, {ymax}].\n"
            f"  Dataset {lat_dim} range: [{float(lv.min()):.2f}, {float(lv.max()):.2f}]\n"
            f"  Dataset {lon_dim} range: [{float(ds[lon_dim].values.min()):.2f}, "
            f"{float(ds[lon_dim].values.max()):.2f}]\n"
            f"This bbox doesn't overlap the cached file's actual coverage -- the cached "
            f".nc was likely generated for a different bbox/country, or COUNTRY_BBOX "
            f"changed since it was cached. Delete the cached file locally "
            f"(Datasets/{COUNTRY}/) and on the Shared Drive, then re-run this cell to "
            f"pull a fresh grid for the current bbox."
        )
    return subset

# ── A) ERA5 long-term temperature ─────────────────────────────────────────────
_lt_filename = f'era5_temperature_monthly_{LT_START}_{LT_END}_{COUNTRY}.nc'
_lt_local    = os.path.join(LOCAL_CACHE_DIR, _lt_filename)
era5_lt      = None
_pull_from_gee = False

print(f'\n\U0001f50d ERA5 temperature: {_lt_filename}')
if os.path.exists(_lt_local):
    try:
        era5_lt = xr.open_dataset(_lt_local)
        # Proactively check for bbox mismatch immediately after loading from local cache
        era5_lt = subset_to_bbox(era5_lt, xmin, xmax, ymin, ymax)
        print('   \u2705 Found locally.')
    except ValueError as e:
        print(f'   \u26a0\ufe0f  Cached file mismatch detected: {e}')
        print('   Deleting local cached file and trying to pull from GEE.')
        os.remove(_lt_local)
        era5_lt = None # Clear era5_lt to force GEE pull
        _pull_from_gee = True
    except Exception as e:
        print(f'   \u26a0\ufe0f  Error opening local cached file: {e}')
        print('   Trying to pull from GEE.')
        era5_lt = None
        _pull_from_gee = True

# Only try shared drive if local failed and no GEE pull is pending yet
if era5_lt is None and not _pull_from_gee:
    try:
        if try_load_from_drive(_lt_local, _lt_filename):
            era5_lt = xr.open_dataset(_lt_local)
            # Proactively check for bbox mismatch immediately after loading from shared drive
            era5_lt = subset_to_bbox(era5_lt, xmin, xmax, ymin, ymax)
    except ValueError as e:
        print(f'   \u26a0\ufe0f  Cached file mismatch detected in Shared Drive: {e}')
        print('   Deleting local cached copy and trying to pull from GEE.')
        os.remove(_lt_local) # Delete the local copy that was just loaded from drive
        era5_lt = None # Clear era5_lt to force GEE pull
        _pull_from_gee = True
    except Exception as e:
        print(f'   \u26a0\ufe0f  Error opening Shared Drive cached file: {e}')
        print('   Trying to pull from GEE.')
        era5_lt = None
        _pull_from_gee = True

if era5_lt is None or _pull_from_gee:
    print('   \u26a0\ufe0f  Not cached or mismatch — pulling spatial grid from GEE...')
    try:
        _region = ee.Geometry.BBox(xmin, ymin, xmax, ymax)
        _col = (ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
                .filterDate(f'{LT_START}-01-01', f'{LT_END}-12-31')
                .select('temperature_2m'))
        _imgs = _col.toList(_col.size())
        _n    = _col.size().getInfo()
        print(f'   Pulling {_n} monthly images...')
        import ipywidgets as _widgets
        from IPython.display import display as _display_era5
        _era5_prog = _widgets.IntProgress(min=0, max=max(_n, 1), description='Pulling months:', bar_style='info')
        _era5_lbl  = _widgets.Label(value=f'0 / {_n}')
        _display_era5(_widgets.HBox([_era5_prog, _era5_lbl]))

        bands_list, times_list = [], []
        for i in range(_n):
            img   = ee.Image(_imgs.get(i))
            t_str = img.date().format('YYYY-MM-dd').getInfo()
            arr   = img.sampleRectangle(region=_region, defaultValue=0)
            vals  = np.array(arr.get('temperature_2m').getInfo(), dtype=float)
            vals  = np.where(vals > 150, vals - 273.15, vals)
            bands_list.append(vals); times_list.append(pd.Timestamp(t_str))
            _era5_prog.value = i + 1
            _era5_lbl.value = f'{i + 1} / {_n}'
        _stack  = np.stack(bands_list, axis=0)
        _lats   = np.linspace(ymax, ymin, bands_list[0].shape[0])
        _lons   = np.linspace(xmin, xmax, bands_list[0].shape[1])
        era5_lt = xr.Dataset(
            {'temperature_2m': (['time', 'lat', 'lon'], _stack)},
            coords={'time': times_list, 'lat': _lats, 'lon': _lons})
        era5_lt.to_netcdf(_lt_local)
        cache_to_drive(_lt_local, _lt_filename)
        print(f'   \u2705 Pulled: {_stack.shape}')
    except Exception as e:
        print(f'   \u274c GEE pull failed: {e}')

if era5_lt is not None:
    _da     = era5_lt[list(era5_lt.data_vars)[0]]
    print(f'   Shape: {_da.shape} | {str(_da.time.values[0])[:10]} \u2192 {str(_da.time.values[-1])[:10]}')
else:
    print('   \u274c ERA5 temperature data could not be loaded or pulled.')

# ── B) Satellite precipitation ────────────────────────────────────────────────
precip_ds        = None
_precip_filename = None
print(f'\n\U0001f50d {PRECIP_SOURCE} precipitation...')

if DRIVE_CACHE_AVAILABLE:
    _matching = [f for f in os.listdir(COUNTRY_DRIVE_DIR)
                 if PRECIP_SOURCE.upper() in f.upper() and f.endswith('.nc')]
    if _matching:
        _precip_filename = _matching[0]
        _precip_local    = os.path.join(LOCAL_CACHE_DIR, _precip_filename)
        _pull_precip_from_gee = False # Flag for precipitation, similar to ERA5

        # If local file does not exist, copy from drive. This is done before the try-except to ensure _precip_local exists for the check.
        if not os.path.exists(_precip_local):
            try:
                shutil.copy(os.path.join(COUNTRY_DRIVE_DIR, _precip_filename), _precip_local)
            except Exception as e:
                print(f'   \u26a0\ufe0f  Error copying precipitation file from Shared Drive: {e}')
                _pull_precip_from_gee = True

        if not _pull_precip_from_gee: # Only proceed if copy was successful or file already existed
            try:
                precip_ds = xr.open_dataset(_precip_local)
                precip_ds = subset_to_bbox(precip_ds, xmin, xmax, ymin, ymax)
                print(f'   \u2705 Found: {_precip_filename}')
            except ValueError as e:
                print(f'   \u26a0\ufe0f  Cached precipitation file mismatch detected: {e}')
                print('   Deleting local cached precipitation file.')
                os.remove(_precip_local)
                precip_ds = None # Clear precip_ds to indicate invalid data
                _pull_precip_from_gee = True
            except Exception as e:
                print(f'   \u26a0\ufe0f  Error opening cached precipitation file: {e}')
                precip_ds = None
                _pull_precip_from_gee = True

        if _pull_precip_from_gee:
            print('   \u26a0\ufe0f  Precipitation data needs to be re-pulled from GEE (This typically requires re-running a prior notebook, like FA2, for fresh cached data).')
            precip_ds = None # Ensure it's cleared if it was invalid
    else:
        print(f'   \u26a0\ufe0f  No {PRECIP_SOURCE} .nc in Shared Drive/{COUNTRY}/')
        print(f'   Available: {os.listdir(COUNTRY_DRIVE_DIR)}')

print(f'\n\U0001f4ca Summary:')
print(f'   ERA5 temp : {"\u2705 spatial grid" if era5_lt is not None else "\u274c missing"}')
print(f'   Precip    : {"\u2705 " + str(_precip_filename) if precip_ds is not None else "\u274c missing"}')

In [ ]:
# @title ### 3b) Load ground station validation data (from Notebook 1 & 3) {"display-mode":"form"}
# @markdown Pulls the resolved ground station data Notebook 1 (precipitation) and
# @markdown Notebook 3 (temperature) already validated for this country -- whichever
# @markdown source ended up being used there (TAHMO/GHCN/Custom) -- to overlay real
# @markdown station locations and readings on the AEZ maps in Step 5b, as a visual
# @markdown sanity check.
# @markdown
# @markdown **This is NOT used as an input to the PyAEZ classification itself.**
# @markdown Ground stations are point data; PyAEZ needs full spatial grids, which only
# @markdown the ERA5/satellite data loaded in Step 3 can provide. This cell is purely
# @markdown for visual validation/context.

import json
import pandas as pd

ground_precip_metadata = pd.DataFrame()
ground_precip_source   = None

ground_temp_metadata = pd.DataFrame()
ground_temp_source   = None

# ── Precipitation ground truth (from Notebook 1) ──────────────────────────────
_precip_meta_path = os.path.join(LOCAL_CACHE_DIR, 'Resolved_Ground_Metadata.csv')
_precip_src_path  = os.path.join(LOCAL_CACHE_DIR, 'Resolved_Ground_Source.json')

if try_load_from_drive(_precip_meta_path, 'Resolved_Ground_Metadata.csv'):
    ground_precip_metadata = pd.read_csv(_precip_meta_path)
    if try_load_from_drive(_precip_src_path, 'Resolved_Ground_Source.json'):
        with open(_precip_src_path) as f:
            ground_precip_source = json.load(f).get('source')
    print(f"✅ Precipitation ground truth loaded from Notebook 1 "
          f"({ground_precip_source or 'unknown source'}, {len(ground_precip_metadata)} station(s)).")
else:
    print("ℹ️  No precipitation ground truth found from Notebook 1 for this country "
          "(run Notebook 1 first if you want the validation overlay).")

# ── Temperature ground truth (from Notebook 3) ────────────────────────────────
_temp_meta_path = os.path.join(LOCAL_CACHE_DIR, 'Resolved_Ground_Temp_Metadata.csv')
_temp_src_path  = os.path.join(LOCAL_CACHE_DIR, 'Resolved_Ground_Temp_Source.json')

if try_load_from_drive(_temp_meta_path, 'Resolved_Ground_Temp_Metadata.csv'):
    ground_temp_metadata = pd.read_csv(_temp_meta_path)
    if try_load_from_drive(_temp_src_path, 'Resolved_Ground_Temp_Source.json'):
        with open(_temp_src_path) as f:
            ground_temp_source = json.load(f).get('source')
    print(f"✅ Temperature ground truth loaded from Notebook 3 "
          f"({ground_temp_source or 'unknown source'}, {len(ground_temp_metadata)} station(s)).")
else:
    print("ℹ️  No temperature ground truth found from Notebook 3 for this country "
          "(run Notebook 3 first if you want the validation overlay).")

print("\nNote: ground stations will appear as markers on the LGP, Thermal, and "
      "Precipitation maps in Step 5b, wherever this data was found.")


In [ ]:
# @title ### 4) Prepare monthly climate arrays {"display-mode":"form"}
# @markdown Produces spatial grids (12, nrows, ncols) and 1D country means (12,).

try:
    advance_notebook_progress(4)
except NameError:
    pass  # overall progress tracker not initialised (Config cell not yet run) -- safe to ignore

MONTHS         = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
days_per_month = np.array([31,28,31,30,31,30,31,31,30,31,30,31])

# ── TEMPERATURE ───────────────────────────────────────────────────────────────
print('Preparing temperature arrays...')

if era5_lt is not None:
    _da = era5_lt[list(era5_lt.data_vars)[0]]
    if float(_da.mean()) > 100: _da = _da - 273.15
    _clim_da  = _da.resample(time='1ME').mean().groupby('time.month').mean('time')
    tavg_clim = _clim_da.values   # (12, nrows, ncols)

    _lat_dim = next((c for c in era5_lt.coords if 'lat' in c.lower()), None)
    _lon_dim = next((c for c in era5_lt.coords if 'lon' in c.lower()), None)
    if _lat_dim and _lon_dim:
        map_lats    = era5_lt[_lat_dim].values
        map_lons    = era5_lt[_lon_dim].values
        HAS_SPATIAL = tavg_clim.ndim == 3
    else:
        HAS_SPATIAL = False

    tavg_monthly_1d = (np.nanmean(tavg_clim, axis=(1, 2)) if HAS_SPATIAL
                       else tavg_clim.flatten()[:12])

    DIURNAL_RANGE   = 10.0
    tmin_clim       = tavg_clim - DIURNAL_RANGE / 2
    tmax_clim       = tavg_clim + DIURNAL_RANGE / 2
    tmin_monthly_1d = tavg_monthly_1d - DIURNAL_RANGE / 2
    tmax_monthly_1d = tavg_monthly_1d + DIURNAL_RANGE / 2

    print(f'   \u2705 Spatial shape: {tavg_clim.shape}  |  HAS_SPATIAL={HAS_SPATIAL}')
    print(f'   \u2705 Monthly Tavg (\u00b0C): {np.round(tavg_monthly_1d, 1)}')
else:
    print('\u26a0\ufe0f  ERA5 not available — using East Africa typical values.')
    tavg_monthly_1d = np.array([22,23,24,24,22,19,18,19,21,22,22,21], dtype=float)
    tmin_monthly_1d = tavg_monthly_1d - 5.0
    tmax_monthly_1d = tavg_monthly_1d + 5.0
    tavg_clim = tmin_clim = tmax_clim = None
    HAS_SPATIAL = False
    map_lats = map_lons = None

# ── PRECIPITATION ─────────────────────────────────────────────────────────────
print('\nPreparing precipitation arrays...')
precip_clim        = None
precip_monthly_1d  = None
HAS_SPATIAL_PRECIP = False

if precip_ds is not None:
    _pda = precip_ds[list(precip_ds.data_vars)[0]]
    _p_clim_da      = _pda.resample(time='1ME').sum().groupby('time.month').mean('time')
    precip_clim_raw = _p_clim_da.values
    n_months        = precip_clim_raw.shape[0]

    if n_months < 12:
        _t0 = str(precip_ds.time.values[0])[:10]
        _t1 = str(precip_ds.time.values[-1])[:10]
        print(f'   \u2139\ufe0f  Precipitation covers {n_months} months ({_t0} \u2192 {_t1})')
        _known_months_idx = [int(m) - 1 for m in _p_clim_da['month'].values]
        _shape            = (12,) + precip_clim_raw.shape[1:]
        precip_clim_full  = np.full(_shape, np.nan)
        for ki, i in zip(_known_months_idx, range(len(_known_months_idx))):
            precip_clim_full[ki] = precip_clim_raw[i]
        _known_mean = np.nanmean(precip_clim_full)
        _missing    = [i for i in range(12) if np.all(np.isnan(precip_clim_full[i]))]
        for mi in _missing:
            precip_clim_full[mi] = _known_mean
        precip_clim = precip_clim_full
        print(f'   \u2705 Known months   : {[i+1 for i in _known_months_idx]}')
        print(f'   \u2705  Estimated months: {[i+1 for i in _missing]} (mean={_known_mean:.1f} mm)')
    else:
        precip_clim = precip_clim_raw

    if precip_clim.ndim > 1:
        precip_monthly_1d  = np.nanmean(precip_clim, axis=tuple(range(1, precip_clim.ndim)))
        HAS_SPATIAL_PRECIP = True
        if map_lats is None:
            _plat = next((c for c in precip_ds.coords if 'lat' in c.lower()), None)
            _plon = next((c for c in precip_ds.coords if 'lon' in c.lower()), None)
            if _plat and _plon:
                map_lats = precip_ds[_plat].values
                map_lons = precip_ds[_plon].values
    else:
        precip_monthly_1d = precip_clim
    print(f'   \u2705 Monthly precip (mm): {np.round(precip_monthly_1d, 1)}')
else:
    print('\u26a0\ufe0f  Precipitation not available — using East Africa typical values.')
    precip_monthly_1d = np.array([40,50,90,120,100,30,15,20,50,90,80,45], dtype=float)

# ── HARGREAVES PET (FAO-56) ───────────────────────────────────────────────────
print('\nComputing monthly PET (Hargreaves-Samani, FAO-56)...')
lat_c      = (ymin + ymax) / 2
Ra_monthly = None

if PYAEZ_AVAILABLE and UtilityCalculations is not None:
    try:
        if hasattr(UtilityCalculations, 'calcRa'):
            Ra_monthly = np.array(
                [float(UtilityCalculations.calcRa(lat_c, m)) for m in range(1, 13)])
        else:
            _cls_list = [x for x in dir(UtilityCalculations)
                         if not x.startswith('_')
                         and isinstance(getattr(UtilityCalculations, x, None), type)]
            if _cls_list:
                _uc_inst   = getattr(UtilityCalculations, _cls_list[0])()
                Ra_monthly = np.array(
                    [float(_uc_inst.calcRa(lat_c, m)) for m in range(1, 13)])
    except Exception:
        Ra_monthly = None   # fall through silently — approximation below is fine

if Ra_monthly is None:
    Ra_monthly = 15.0 + 3.0 * np.cos(np.linspace(0, 2 * np.pi, 12) + np.pi / 6)
    print(f'   \u2705 Ra (latitude approx, lat={lat_c:.1f}\u00b0): {np.round(Ra_monthly, 1)}')
else:
    print(f'   \u2705 Ra from PyAEZ: {np.round(Ra_monthly, 1)}')

pet_monthly_1d = (
    0.0023 * Ra_monthly
    * (tavg_monthly_1d + 17.8)
    * np.sqrt(np.maximum(tmax_monthly_1d - tmin_monthly_1d, 0.1))
)
pet_monthly_mm = pet_monthly_1d * days_per_month

if HAS_SPATIAL and tavg_clim is not None:
    pet_clim = np.zeros_like(tavg_clim)
    for m in range(12):
        _td = np.sqrt(np.maximum(tmax_clim[m] - tmin_clim[m], 0.1))
        pet_clim[m] = (0.0023 * Ra_monthly[m] * (tavg_clim[m] + 17.8)
                       * _td * days_per_month[m])
    print(f'   \u2705 Spatial PET grid: {pet_clim.shape}')
else:
    pet_clim = None

water_balance_monthly = precip_monthly_1d - pet_monthly_mm
print(f'\n   PET (mm/month):        {np.round(pet_monthly_mm, 0)}')
print(f'   Water balance (P-PET): {np.round(water_balance_monthly, 0)}')

# ── SPATIAL AEZ MAPS (manual — reliable fallback always computed) ──────────────
print('\n\U0001f5fa\ufe0f  Computing spatial AEZ maps...')

if HAS_SPATIAL and tavg_clim is not None:
    _nrows_s, _ncols_s = tavg_clim.shape[1], tavg_clim.shape[2]

    _pet_spatial = pet_clim if pet_clim is not None else np.stack([
        0.0023 * Ra_monthly[m] * (tavg_clim[m] + 17.8)
        * np.sqrt(np.maximum(tmax_clim[m] - tmin_clim[m], 0.1)) * days_per_month[m]
        for m in range(12)], axis=0)

    if precip_clim is not None and precip_clim.ndim == 3:
        from scipy.interpolate import RegularGridInterpolator
        _plat   = next((c for c in precip_ds.coords if 'lat' in c.lower()), None)
        _plon   = next((c for c in precip_ds.coords if 'lon' in c.lower()), None)
        _p_lats = (precip_ds[_plat].values if _plat
                   else np.linspace(ymin, ymax, precip_clim.shape[1]))
        _p_lons = (precip_ds[_plon].values if _plon
                   else np.linspace(xmin, xmax, precip_clim.shape[2]))
        _prec_on_era5 = np.zeros((12, _nrows_s, _ncols_s))
        for m in range(12):
            try:
                _interp = RegularGridInterpolator(
                    (np.sort(_p_lats), np.sort(_p_lons)),
                    precip_clim[m] if _p_lats[0] < _p_lats[-1] else precip_clim[m][::-1],
                    method='linear', bounds_error=False, fill_value=np.nan)
                _gl, _glo = np.meshgrid(map_lats, map_lons, indexing='ij')
                _prec_on_era5[m] = _interp(np.stack([_gl, _glo], axis=-1))
            except Exception:
                _prec_on_era5[m] = precip_monthly_1d[m]
        precip_spatial = _prec_on_era5
    else:
        precip_spatial = (precip_monthly_1d[:, np.newaxis, np.newaxis]
                          * np.ones((12, _nrows_s, _ncols_s)))

    lgp_map = ((precip_spatial >= 0.5 * _pet_spatial).sum(axis=0) * 30).astype(float)

    _t_min_g  = tavg_clim.min(axis=0)
    _t_mean_g = tavg_clim.mean(axis=0)
    _cold_m   = (tavg_clim < 5).sum(axis=0)
    thermal_map = np.zeros((_nrows_s, _ncols_s), dtype=float)
    thermal_map[_t_min_g >= 20]                                              = 1
    thermal_map[(_t_min_g >= 17) & (_t_min_g < 20) & (_cold_m == 0)]       = 2
    thermal_map[(_t_mean_g >= 15) & (_cold_m < 3)  & (thermal_map == 0)]   = 3
    thermal_map[(_t_mean_g >= 10) & (_cold_m < 6)  & (thermal_map == 0)]   = 4
    thermal_map[(_t_mean_g >= 5)  & (thermal_map == 0)]                     = 5
    thermal_map[(_t_mean_g >= 0)  & (thermal_map == 0)]                     = 6
    thermal_map[thermal_map == 0]                                             = 7

    moisture_map = np.zeros((_nrows_s, _ncols_s), dtype=float)
    moisture_map[lgp_map < 30]                           = 1
    moisture_map[(lgp_map >= 30)  & (lgp_map < 60)]     = 2
    moisture_map[(lgp_map >= 60)  & (lgp_map < 120)]    = 3
    moisture_map[(lgp_map >= 120) & (lgp_map < 180)]    = 4
    moisture_map[(lgp_map >= 180) & (lgp_map < 270)]    = 5
    moisture_map[lgp_map >= 270]                          = 6
    aez_map          = thermal_map * 10 + moisture_map
    HAS_SPATIAL_MAPS = True

    print(f'   \u2705 LGP map      : min={lgp_map.min():.0f}  max={lgp_map.max():.0f}  mean={lgp_map.mean():.0f} days')
    print(f'   \u2705 Thermal map  : classes {np.unique(thermal_map).astype(int).tolist()}')
    print(f'   \u2705 Moisture map : zones {np.unique(moisture_map).astype(int).tolist()}')
else:
    lgp_map = thermal_map = moisture_map = aez_map = None
    precip_spatial = _pet_spatial = None
    HAS_SPATIAL_MAPS = False
    print('   \u26a0\ufe0f  No spatial grid — maps skipped. Charts still work.')

In [ ]:
# @title ### 5) Module I — Climate Regime Analysis {"display-mode":"form"}
# @markdown PyAEZ v2.2 ClimateRegime with runtime parameter discovery.
# @markdown Falls back to manually computed maps from Cell 4 if PyAEZ fails.

try:
    advance_notebook_progress(5)
except NameError:
    pass  # overall progress tracker not initialised (Config cell not yet run) -- safe to ignore

THERMAL_CLASSES = {
    1: 'Tropics, lowland    (Tavg \u2265 20\u00b0C all months)',
    2: 'Tropics, subtropics (Tavg 17\u201320\u00b0C, no frost)',
    3: 'Subtropics, warm    (Tavg 10\u201320\u00b0C, < 3 cold months)',
    4: 'Subtropics, cool    (Tavg 5\u201315\u00b0C, 3\u20136 cold months)',
    5: 'Temperate, moderate (Tavg 0\u201310\u00b0C, frost present)',
    6: 'Temperate, cool/cold (Tavg < 5\u00b0C in winter)',
    7: 'Boreal / Arctic',
}
MOISTURE_CLASSES = {
    1: 'Hyper-arid    (LGP < 30 days)',
    2: 'Arid          (LGP 30\u201360 days)',
    3: 'Semi-arid     (LGP 60\u2013120 days)',
    4: 'Sub-humid     (LGP 120\u2013180 days)',
    5: 'Humid         (LGP 180\u2013270 days)',
    6: 'Per-humid     (LGP > 270 days)',
}

lgp_value = thermal_class = moisture_class = None

# ── PyAEZ v2.2 spatial run ────────────────────────────────────────────────────
if PYAEZ_AVAILABLE and HAS_SPATIAL_MAPS and tavg_clim is not None:
    try:
        _nrows_s, _ncols_s = tavg_clim.shape[1], tavg_clim.shape[2]
        _mask = np.ones((_nrows_s, _ncols_s), dtype=int)

        cr = ClimateRegime.ClimateRegime()
        cr.setStudyAreaMask(_mask, no_data_value=0)

        # FIX 1: transpose (12,r,c) → (r,c,12) as required by v2.2
        _T = lambda a: np.transpose(a, (1, 2, 0))

        # FIX 2: discover actual parameter names at runtime via inspect
        #         so the call never fails due to wrong keyword names
        _sig    = inspect.signature(cr.setMonthlyClimateData)
        _params = list(_sig.parameters.keys())

        _call_kwargs = {}
        for p in _params:
            pl = p.lower()
            if   'min'    in pl and 'temp' in pl:                          _call_kwargs[p] = _T(tmin_clim)
            elif 'max'    in pl and 'temp' in pl:                          _call_kwargs[p] = _T(tmax_clim)
            elif ('mean'  in pl or 'avg' in pl or
                  ('temp' in pl and 'min' not in pl and 'max' not in pl)): _call_kwargs[p] = _T(tavg_clim)
            elif 'precip' in pl or 'rain' in pl:                           _call_kwargs[p] = _T(precip_spatial)
            elif 'rad'    in pl or 'solar' in pl or 'short' in pl:         _call_kwargs[p] = None
            elif 'wind'   in pl:                                            _call_kwargs[p] = None

        cr.setMonthlyClimateData(**_call_kwargs)

        cr.getThermalClimate()
        cr.getLGP(Sa=100, D50=0.5)
        cr.getThermalZone()

        _lgp_out     = np.array(cr.lgp,            dtype=float)
        _thermal_out = np.array(cr.thermal_climate, dtype=float)

        # FIX 3: getMoistureZone() removed in v2.2 — derive from LGP (FAO thresholds)
        def _lgp_to_mz(a):
            mz = np.zeros_like(a)
            mz[a < 30]                           = 1
            mz[(a >= 30)  & (a < 60)]           = 2
            mz[(a >= 60)  & (a < 120)]          = 3
            mz[(a >= 120) & (a < 180)]          = 4
            mz[(a >= 180) & (a < 270)]          = 5
            mz[a >= 270]                          = 6
            return mz

        _moisture_out = _lgp_to_mz(_lgp_out)

        lgp_map      = _lgp_out
        thermal_map  = _thermal_out
        moisture_map = _moisture_out
        aez_map      = thermal_map * 10 + moisture_map

        lgp_value      = int(np.nanmean(lgp_map))
        thermal_class  = int(np.nanmedian(thermal_map))
        moisture_class = int(np.nanmedian(moisture_map))

        print('\u2705 PyAEZ ClimateRegime ran on FULL SPATIAL GRID.')
        print(f'   LGP map       : min={lgp_map.min():.0f}  max={lgp_map.max():.0f}  mean={lgp_map.mean():.0f} days')
        print(f'   Thermal zones : {np.unique(thermal_map[~np.isnan(thermal_map)]).astype(int).tolist()}')
        print(f'   Moisture zones: {np.unique(moisture_map[~np.isnan(moisture_map)]).astype(int).tolist()}')

    except Exception as e:
        print('   Using manually computed maps')

# ── Fallback scalars from manual maps ─────────────────────────────────────────
if lgp_value is None:
    if HAS_SPATIAL_MAPS and lgp_map is not None:
        lgp_value      = int(np.nanmean(lgp_map))
        thermal_class  = int(np.nanmedian(thermal_map))
        moisture_class = int(np.nanmedian(moisture_map))
    else:
        def _lgp1d(p, pet): return int((p >= 0.5 * pet).sum() * 30)
        def _therm(t):
            if t.min()  >= 20: return 1
            if t.min()  >= 17: return 2
            if t.mean() >= 15: return 3
            if t.mean() >= 10: return 4
            if t.mean() >=  5: return 5
            if t.mean() >=  0: return 6
            return 7
        def _moist(lgp):
            for thr, cls in [(30,1),(60,2),(120,3),(180,4),(270,5)]:
                if lgp < thr: return cls
            return 6
        lgp_value      = _lgp1d(precip_monthly_1d, pet_monthly_mm)
        thermal_class  = _therm(tavg_monthly_1d)
        moisture_class = _moist(lgp_value)

tavg_annual   = tavg_monthly_1d.mean()
precip_annual = precip_monthly_1d.sum()

print('\n' + '\u2550' * 60)
print(f'  CLIMATE REGIME RESULTS \u2014 {COUNTRY}')
print('\u2550' * 60)
print(f'  Thermal class   : {thermal_class} \u2014 {THERMAL_CLASSES.get(thermal_class, "Unknown")}')
print(f'  LGP (mean)      : {lgp_value} days/year')
print(f'  Moisture zone   : {moisture_class} \u2014 {MOISTURE_CLASSES.get(moisture_class, "Unknown")}')
print(f'  Annual Tavg     : {tavg_annual:.1f}\u00b0C')
print(f'  Annual Precip   : {precip_annual:.0f} mm')
print(f'  Annual PET      : {pet_monthly_mm.sum():.0f} mm')
print(f'  Aridity index   : {precip_annual / pet_monthly_mm.sum():.2f} (P/PET)')
print(f'  Spatial maps    : {"\u2705 available" if HAS_SPATIAL_MAPS else "\u26a0\ufe0f  1D mode"}')
print('\u2550' * 60)

In [ ]:
# @title ### 5b) Geospatial AEZ Maps {"display-mode":"form"}
# @markdown LGP · Thermal · Moisture · Precipitation · AEZ composite · Water balance

if not HAS_SPATIAL_MAPS or lgp_map is None:
    print('\u26a0\ufe0f  Spatial maps not available.')
    print('   Re-run Cell 3 to pull the spatial grid from GEE and cache it.')
else:
    def _overlay_ground_stations(ax, metadata, color, label):
        """Plots ground station locations from Notebook 1/3's resolved metadata
        on top of a map, as a visual validation layer -- these are NOT inputs to
        the classification, just real observation points for a sanity check."""
        if metadata is None or metadata.empty:
            return
        _lat_col = next((c for c in metadata.columns if 'lat' in c.lower()), None)
        _lon_col = next((c for c in metadata.columns
                         if 'lon' in c.lower() or 'lng' in c.lower()), None)
        if not _lat_col or not _lon_col:
            return
        ax.scatter(metadata[_lon_col], metadata[_lat_col],
                   s=45, c=color, edgecolors='black', linewidths=0.8,
                   marker='o', zorder=5, label=label)

    fig = plt.figure(figsize=(20, 16))
    fig.suptitle(
        f'Focus Area 4 \u2014 Agro-Ecological Zone Maps\n'
        f'{COUNTRY}  |  {LT_START}\u2013{LT_END}',
        fontsize=15, fontweight='bold', y=0.99)

    _extent = [xmin, xmax, ymin, ymax]

    # MAP 1: LGP ──────────────────────────────────────────────────────────────
    ax1 = fig.add_subplot(2, 3, 1)
    im1 = ax1.imshow(lgp_map, extent=_extent, cmap=plt.cm.RdYlGn,
                     norm=mcolors.Normalize(vmin=0, vmax=365),
                     origin='upper', aspect='auto', interpolation='bilinear')
    plt.colorbar(im1, ax=ax1, label='Days/year', shrink=0.8, pad=0.02)
    ax1.set_title(f'Length of Growing Period\nMean = {lgp_map.mean():.0f} days/yr', fontsize=11)
    ax1.set_xlabel('Longitude'); ax1.set_ylabel('Latitude')
    ax1.set_xlim(xmin, xmax); ax1.set_ylim(ymin, ymax)
    ax1.grid(True, alpha=0.3, color='white', linewidth=0.5)
    _overlay_ground_stations(ax1, ground_precip_metadata, 'blue', 'Ground precip stations')
    if 'ground_precip_metadata' in globals() and not ground_precip_metadata.empty:
        ax1.legend(loc='upper right', fontsize=7, framealpha=0.8)
    try:
        _cs = ax1.contour(
            np.linspace(xmin, xmax, lgp_map.shape[1]),
            np.linspace(ymin, ymax, lgp_map.shape[0]),
            lgp_map, levels=[60, 120, 180, 270],
            colors='white', linewidths=0.8, alpha=0.7)
        ax1.clabel(_cs, fmt='%d d', fontsize=7, colors='white')
    except Exception:
        pass

    # MAP 2: Thermal Zone ─────────────────────────────────────────────────────
    ax2 = fig.add_subplot(2, 3, 2)
    _t_colors = ['#d73027','#fc8d59','#fee08b','#d9ef8b','#91cf60','#1a9850','#4575b4']
    _tm       = max(1, int(np.nanmax(thermal_map)))
    _t_cmap   = mcolors.ListedColormap(_t_colors[:_tm])
    _t_norm   = mcolors.BoundaryNorm(np.arange(0.5, _tm + 1.5), _t_cmap.N)
    ax2.imshow(thermal_map, extent=_extent, cmap=_t_cmap, norm=_t_norm,
               origin='upper', aspect='auto', interpolation='nearest')
    _t_lbl = {1:'Tropics low',2:'Tropics/sub',3:'Subtrop warm',
              4:'Subtrop cool',5:'Temperate',6:'Temp cool',7:'Boreal'}
    ax2.legend(handles=[
        mpatches.Patch(color=_t_colors[int(v)-1],
                       label=f'Class {int(v)}: {_t_lbl.get(int(v),"?")}')
        for v in np.unique(thermal_map) if not np.isnan(v)],
        loc='lower left', fontsize=7, framealpha=0.8)
    ax2.set_title('Thermal Climate Classification', fontsize=11)
    ax2.set_xlabel('Longitude'); ax2.set_ylabel('Latitude')
    ax2.set_xlim(xmin, xmax); ax2.set_ylim(ymin, ymax)
    ax2.grid(True, alpha=0.3)
    _overlay_ground_stations(ax2, ground_temp_metadata, 'cyan', 'Ground temp stations')

    # MAP 3: Moisture Zone ────────────────────────────────────────────────────
    ax3 = fig.add_subplot(2, 3, 3)
    _m_colors = ['#d73027','#f46d43','#fdae61','#fee090','#74add1','#313695']
    _mm       = max(1, int(np.nanmax(moisture_map)))
    _m_cmap   = mcolors.ListedColormap(_m_colors[:_mm])
    _m_norm   = mcolors.BoundaryNorm(np.arange(0.5, _mm + 1.5), _m_cmap.N)
    ax3.imshow(moisture_map, extent=_extent, cmap=_m_cmap, norm=_m_norm,
               origin='upper', aspect='auto', interpolation='nearest')
    _m_lbl = {1:'Hyper-arid (<30d)',2:'Arid (30\u201360d)',3:'Semi-arid (60\u2013120d)',
              4:'Sub-humid (120\u2013180d)',5:'Humid (180\u2013270d)',6:'Per-humid (>270d)'}
    ax3.legend(handles=[
        mpatches.Patch(color=_m_colors[int(v)-1], label=_m_lbl.get(int(v),'?'))
        for v in np.unique(moisture_map) if not np.isnan(v)],
        loc='lower left', fontsize=7, framealpha=0.8)
    ax3.set_title('Moisture Zone Classification', fontsize=11)
    ax3.set_xlabel('Longitude'); ax3.set_ylabel('Latitude')
    ax3.set_xlim(xmin, xmax); ax3.set_ylim(ymin, ymax)
    ax3.grid(True, alpha=0.3)

    # MAP 4: Annual Precipitation ─────────────────────────────────────────────
    ax4 = fig.add_subplot(2, 3, 4)
    _ann_p = (precip_clim.sum(axis=0)
              if precip_clim is not None and precip_clim.ndim == 3
              else np.full(lgp_map.shape, precip_monthly_1d.sum()))
    if _ann_p.shape != lgp_map.shape:
        _ann_p = np.full(lgp_map.shape, float(np.nanmean(_ann_p)))
    im4 = ax4.imshow(_ann_p, extent=_extent, cmap=plt.cm.YlGnBu,
                     norm=mcolors.Normalize(vmin=0, vmax=max(float(np.nanmax(_ann_p)), 1000)),
                     origin='upper', aspect='auto', interpolation='bilinear')
    plt.colorbar(im4, ax=ax4, label='mm/year', shrink=0.8, pad=0.02)
    ax4.set_title(f'Annual Precipitation ({PRECIP_SOURCE})\nMean = {precip_monthly_1d.sum():.0f} mm/yr', fontsize=11)
    ax4.set_xlabel('Longitude'); ax4.set_ylabel('Latitude')
    ax4.set_xlim(xmin, xmax); ax4.set_ylim(ymin, ymax)
    ax4.grid(True, alpha=0.3)
    _overlay_ground_stations(ax4, ground_precip_metadata, 'blue', 'Ground precip stations')

    # MAP 5: AEZ Composite ────────────────────────────────────────────────────
    ax5 = fig.add_subplot(2, 3, 5)
    im5 = ax5.imshow(aez_map, extent=_extent, cmap=plt.cm.tab20,
                     norm=mcolors.Normalize(vmin=float(np.nanmin(aez_map)),
                                            vmax=float(np.nanmax(aez_map))),
                     origin='upper', aspect='auto', interpolation='nearest')
    plt.colorbar(im5, ax=ax5, label='Thermal\u00d710 + Moisture', shrink=0.8, pad=0.02)
    ax5.set_title('AEZ Composite\n(Thermal \u00d7 10 + Moisture Zone)', fontsize=11)
    ax5.set_xlabel('Longitude'); ax5.set_ylabel('Latitude')
    ax5.set_xlim(xmin, xmax); ax5.set_ylim(ymin, ymax)
    ax5.grid(True, alpha=0.3)

    # MAP 6: Annual Water Balance — SAFE shape handling ───────────────────────
    ax6  = fig.add_subplot(2, 3, 6)
    _wb_spatial = None

    if precip_clim is not None and precip_clim.ndim == 3 and _pet_spatial is not None:
        _p_ann = precip_clim.sum(axis=0)
        _e_ann = _pet_spatial.sum(axis=0)

        if _p_ann.shape == _e_ann.shape:
            _wb_spatial = _p_ann - _e_ann
        else:
            # Different grids — regrid PET onto the precip grid
            try:
                from scipy.interpolate import RegularGridInterpolator
                _plat_wb  = next((c for c in precip_ds.coords if 'lat' in c.lower()), None)
                _plon_wb  = next((c for c in precip_ds.coords if 'lon' in c.lower()), None)
                _p_lats_wb = (precip_ds[_plat_wb].values if _plat_wb
                              else np.linspace(ymin, ymax, _p_ann.shape[0]))
                _p_lons_wb = (precip_ds[_plon_wb].values if _plon_wb
                              else np.linspace(xmin, xmax, _p_ann.shape[1]))
                _interp_wb = RegularGridInterpolator(
                    (np.sort(map_lats), np.sort(map_lons)),
                    _e_ann if map_lats[0] < map_lats[-1] else _e_ann[::-1],
                    method='linear', bounds_error=False,
                    fill_value=float(np.nanmean(_e_ann)))
                _gl_wb, _glo_wb = np.meshgrid(
                    np.sort(_p_lats_wb), np.sort(_p_lons_wb), indexing='ij')
                _e_ann_rg   = _interp_wb(np.stack([_gl_wb, _glo_wb], axis=-1))
                _wb_spatial = _p_ann - _e_ann_rg
            except Exception:
                _wb_spatial = None

    if _wb_spatial is None:
        _wb_spatial = np.full(lgp_map.shape, float(water_balance_monthly.sum()))

    # Ensure shape matches lgp_map for imshow extent
    if _wb_spatial.shape != lgp_map.shape:
        _wb_spatial = np.full(lgp_map.shape, float(np.nanmean(_wb_spatial)))

    _wb_max = max(abs(float(np.nanmin(_wb_spatial))),
                  abs(float(np.nanmax(_wb_spatial))), 100)
    im6 = ax6.imshow(_wb_spatial, extent=_extent, cmap=plt.cm.RdBu,
                     norm=mcolors.Normalize(vmin=-_wb_max, vmax=_wb_max),
                     origin='upper', aspect='auto', interpolation='bilinear')
    plt.colorbar(im6, ax=ax6, label='mm/year', shrink=0.8, pad=0.02)
    ax6.set_title('Annual Water Balance (P \u2212 PET)', fontsize=11)
    ax6.set_xlabel('Longitude'); ax6.set_ylabel('Latitude')
    ax6.set_xlim(xmin, xmax); ax6.set_ylim(ymin, ymax)
    ax6.grid(True, alpha=0.3)

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    _map_path = os.path.join(RESULTS_DIR, f'FA4_AEZ_{COUNTRY}.png')
    plt.savefig(_map_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\u2705 Map saved: {_map_path}')
    print(f'   LGP range      : {lgp_map.min():.0f}\u2013{lgp_map.max():.0f} days/yr')
    print(f'   Thermal zones  : {sorted(set(int(v) for v in np.unique(thermal_map) if not np.isnan(v)))}')
    print(f'   Moisture zones : {sorted(set(int(v) for v in np.unique(moisture_map) if not np.isnan(v)))}')

In [ ]:
# @title ### 6) Crop suitability assessment {"display-mode":"form"}
# @markdown FAO GAEZ v4 crop parameter tables — East Africa + Horn of Africa crops.

try:
    advance_notebook_progress(6)
except NameError:
    pass  # overall progress tracker not initialised (Config cell not yet run) -- safe to ignore

CROPS = {
    'Maize':        {'tavg_min':10.0,'tavg_max':35.0,'tmin_abs': 5.0,'tmax_abs':40.0,
                     'lgp_min': 90,'lgp_opt':150,'precip_min':400,'precip_opt': 700,
                     'description':'Most important food crop in East Africa.'},
    'Sorghum':      {'tavg_min':12.0,'tavg_max':37.0,'tmin_abs': 6.0,'tmax_abs':42.0,
                     'lgp_min': 75,'lgp_opt':120,'precip_min':250,'precip_opt': 500,
                     'description':'Drought-tolerant. Suited to semi-arid zones.'},
    'Wheat':        {'tavg_min': 5.0,'tavg_max':25.0,'tmin_abs': 0.0,'tmax_abs':32.0,
                     'lgp_min': 90,'lgp_opt':150,'precip_min':300,'precip_opt': 600,
                     'description':'Higher elevations (Ethiopia highlands).'},
    'Millet (Pearl)':{'tavg_min':14.0,'tavg_max':38.0,'tmin_abs': 8.0,'tmax_abs':44.0,
                      'lgp_min': 60,'lgp_opt':100,'precip_min':200,'precip_opt': 400,
                      'description':'Most drought-tolerant cereal. Suited to arid margins.'},
    'Rice (Upland)': {'tavg_min':18.0,'tavg_max':35.0,'tmin_abs':12.0,'tmax_abs':38.0,
                      'lgp_min': 90,'lgp_opt':150,'precip_min':800,'precip_opt':1200,
                      'description':'Humid lowlands and irrigated areas.'},
    'Cassava':      {'tavg_min':18.0,'tavg_max':35.0,'tmin_abs':10.0,'tmax_abs':38.0,
                     'lgp_min':120,'lgp_opt':240,'precip_min':500,'precip_opt':1000,
                     'description':'Resilient root crop. Tolerates poor soils.'},
    'Teff':         {'tavg_min':10.0,'tavg_max':27.0,'tmin_abs': 5.0,'tmax_abs':33.0,
                     'lgp_min': 75,'lgp_opt':120,'precip_min':300,'precip_opt': 600,
                     'description':'Ethiopian staple grain. Tolerates waterlogged or dry soils.'},
    'Sesame':       {'tavg_min':25.0,'tavg_max':38.0,'tmin_abs':15.0,'tmax_abs':45.0,
                     'lgp_min': 75,'lgp_opt':100,'precip_min':300,'precip_opt': 600,
                     'description':'Major oil crop for Sudan, Ethiopia, Somalia.'},
    'Groundnut':    {'tavg_min':22.0,'tavg_max':35.0,'tmin_abs':10.0,'tmax_abs':40.0,
                     'lgp_min': 90,'lgp_opt':140,'precip_min':400,'precip_opt': 700,
                     'description':'Important legume and oil crop across East Africa.'},
}

tavg_annual   = tavg_monthly_1d.mean()
tmin_annual   = tmin_monthly_1d.min()
tmax_annual   = tmax_monthly_1d.max()
precip_annual = precip_monthly_1d.sum()

results = []
for crop, req in CROPS.items():
    thermal_ok = (req['tavg_min'] <= tavg_annual <= req['tavg_max']
                  and tmin_annual >= req['tmin_abs']
                  and tmax_annual <= req['tmax_abs'] + 3)
    lgp_ok   = lgp_value >= req['lgp_min']
    lgp_opt  = lgp_value >= req['lgp_opt']
    water_ok  = precip_annual >= req['precip_min']
    water_opt = precip_annual >= req['precip_opt']

    if thermal_ok and lgp_ok and water_ok:
        suitability, score = ('High', 3) if lgp_opt and water_opt else ('Moderate', 2)
    elif thermal_ok and (lgp_ok or water_ok):
        suitability, score = 'Marginal', 1
    else:
        suitability, score = 'Not suitable', 0

    results.append({
        'Crop':        crop,
        'Suitability': suitability,
        'Score':       score,
        'Thermal OK':  '\u2705' if thermal_ok else '\u274c',
        'LGP OK':      '\u2705' if lgp_ok     else '\u274c',
        'Water OK':    '\u2705' if water_ok   else '\u274c',
        'Description': req['description'],
    })

df_suitability = pd.DataFrame(results).sort_values('Score', ascending=False)

print(f'\nCROP SUITABILITY ASSESSMENT \u2014 {COUNTRY}')
print(f'  Tavg={tavg_annual:.1f}\u00b0C  Tmin={tmin_annual:.1f}\u00b0C  Tmax={tmax_annual:.1f}\u00b0C')
print(f'  LGP={lgp_value} days  Precip={precip_annual:.0f} mm')
print()
display(df_suitability[['Crop','Suitability','Thermal OK','LGP OK','Water OK','Description']])

_csv_path = os.path.join(RESULTS_DIR, f'FA4_crop_suitability_{COUNTRY}.csv')
df_suitability.to_csv(_csv_path, index=False)
print(f'\n\u2705 CSV saved: {_csv_path}')

In [ ]:
# @title ### 7) Results visualisation {"display-mode":"form"}
# @markdown Climate profile · Water balance · PET · Crop suitability · Growing calendar

try:
    advance_notebook_progress(7)
except NameError:
    pass  # overall progress tracker not initialised (Config cell not yet run) -- safe to ignore

fig = plt.figure(figsize=(18, 14))
fig.suptitle(
    f'Focus Area 4 \u2014 Agro-Ecological Zoning\n{COUNTRY}  |  {LT_START}\u2013{LT_END}',
    fontsize=14, fontweight='bold', y=0.98)

# Panel 1: Temperature profile
ax1 = fig.add_subplot(3, 3, 1)
ax1.fill_between(range(12), tmin_monthly_1d, tmax_monthly_1d,
                 alpha=0.2, color='tomato', label='Tmin\u2013Tmax range')
ax1.plot(range(12), tavg_monthly_1d, 'o-', color='tomato', lw=2, ms=5, label='Tavg')
ax1.axhline(10, color='steelblue', ls='--', lw=0.8, label='10\u00b0C (crop base)')
ax1.axhline(0,  color='black',     ls='--', lw=0.8, label='0\u00b0C (frost)')
ax1.set_xticks(range(12)); ax1.set_xticklabels(MONTHS, rotation=45, ha='right', fontsize=8)
ax1.set_ylabel('\u00b0C'); ax1.set_title('Monthly Temperature')
ax1.legend(fontsize=7); ax1.grid(True, alpha=0.3)

# Panel 2: Precipitation vs PET
ax2 = fig.add_subplot(3, 3, 2)
ax2.bar(range(12), precip_monthly_1d, color='steelblue', alpha=0.8, label='Precipitation')
ax2.plot(range(12), pet_monthly_mm, 'o-', color='tomato', lw=2, ms=5, label='PET (Hargreaves)')
ax2.set_xticks(range(12)); ax2.set_xticklabels(MONTHS, rotation=45, ha='right', fontsize=8)
ax2.set_ylabel('mm'); ax2.set_title('Precipitation vs PET')
ax2.legend(fontsize=7); ax2.grid(True, alpha=0.3, axis='y')

# Panel 3: Water balance
ax3 = fig.add_subplot(3, 3, 3)
colors_wb = ['steelblue' if v >= 0 else 'tomato' for v in water_balance_monthly]
ax3.bar(range(12), water_balance_monthly, color=colors_wb, alpha=0.85)
ax3.axhline(0, color='black', lw=1)
ax3.set_xticks(range(12)); ax3.set_xticklabels(MONTHS, rotation=45, ha='right', fontsize=8)
ax3.set_ylabel('mm'); ax3.set_title('Water Balance (P \u2212 PET)')
ax3.grid(True, alpha=0.3, axis='y')

# Panel 4: Crop suitability bar chart
ax4 = fig.add_subplot(3, 3, 4)
_suit_colors = {'High':'#2e7d52','Moderate':'#c87c1a','Marginal':'#d4a017','Not suitable':'#b5341a'}
ax4.barh(df_suitability['Crop'], df_suitability['Score'],
         color=[_suit_colors[s] for s in df_suitability['Suitability']], alpha=0.85)
ax4.set_xlim(0, 3.5)
ax4.set_xticks([0, 1, 2, 3])
ax4.set_xticklabels(['Not\nSuitable','Marginal','Moderate','High'], fontsize=8)
ax4.set_title('Crop Suitability Score'); ax4.grid(True, alpha=0.3, axis='x')

# Panel 5: AEZ summary text card
ax5 = fig.add_subplot(3, 3, 5)
ax5.axis('off')
summary_text = (
    f'AEZ SUMMARY\n'
    f'{"\u2500"*28}\n'
    f'Country       : {COUNTRY}\n'
    f'Thermal class : {thermal_class}\n'
    f'  {THERMAL_CLASSES.get(thermal_class,"?")[:30]}\n\n'
    f'Moisture zone : {moisture_class}\n'
    f'  {MOISTURE_CLASSES.get(moisture_class,"?")[:30]}\n\n'
    f'LGP           : {lgp_value} days/yr\n'
    f'Annual Tavg   : {tavg_annual:.1f}\u00b0C\n'
    f'Annual Precip : {precip_annual:.0f} mm\n'
    f'Annual PET    : {pet_monthly_mm.sum():.0f} mm\n'
    f'Aridity (P/PET): {precip_annual/pet_monthly_mm.sum():.2f}'
)
ax5.text(0.05, 0.95, summary_text, transform=ax5.transAxes, fontsize=9,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# Panel 6: Growing period calendar
ax6 = fig.add_subplot(3, 3, 6)
_growing = (precip_monthly_1d >= 0.5 * pet_monthly_mm).astype(float)
ax6.bar(range(12), _growing * precip_monthly_1d, color='#2e7d52', alpha=0.85, label='Growing')
ax6.bar(range(12), (1 - _growing) * precip_monthly_1d, color='#b5341a', alpha=0.5, label='Dry')
ax6.set_xticks(range(12)); ax6.set_xticklabels(MONTHS, rotation=45, ha='right', fontsize=8)
ax6.set_ylabel('Precip (mm)'); ax6.set_title('Growing vs Dry Months\n(P \u2265 0.5\u00d7PET = growing)')
ax6.legend(fontsize=7); ax6.grid(True, alpha=0.3, axis='y')

# Panel 7: Ra and PET
ax7 = fig.add_subplot(3, 3, 7)
ax7.plot(range(12), Ra_monthly, 's-', color='orange', lw=2, ms=5, label='Ra (MJ/m\u00b2/day)')
ax7r = ax7.twinx()
ax7r.bar(range(12), pet_monthly_mm, color='coral', alpha=0.5, label='PET (mm/month)')
ax7.set_xticks(range(12)); ax7.set_xticklabels(MONTHS, rotation=45, ha='right', fontsize=8)
ax7.set_ylabel('Ra (MJ/m\u00b2/day)'); ax7r.set_ylabel('PET mm/month')
ax7.set_title('Extraterrestrial Radiation & PET')
lines1, labels1 = ax7.get_legend_handles_labels()
lines2, labels2 = ax7r.get_legend_handles_labels()
ax7.legend(lines1 + lines2, labels1 + labels2, fontsize=7, loc='upper right')
ax7.grid(True, alpha=0.3)

# Panel 8: Cumulative water balance
ax8 = fig.add_subplot(3, 3, 8)
_cumwb = np.cumsum(water_balance_monthly)
ax8.plot(range(12), _cumwb, 'o-', color='steelblue', lw=2, ms=5)
ax8.fill_between(range(12), _cumwb, alpha=0.2,
                 color='steelblue' if _cumwb[-1] >= 0 else 'tomato')
ax8.axhline(0, color='black', lw=1)
ax8.set_xticks(range(12)); ax8.set_xticklabels(MONTHS, rotation=45, ha='right', fontsize=8)
ax8.set_ylabel('Cumulative mm'); ax8.set_title('Cumulative Water Balance')
ax8.grid(True, alpha=0.3)

# Panel 9: Suitability pie
ax9 = fig.add_subplot(3, 3, 9)
_counts = df_suitability['Suitability'].value_counts()
_pie_colors = [_suit_colors.get(k, 'grey') for k in _counts.index]
ax9.pie(_counts.values, labels=_counts.index, colors=_pie_colors,
        autopct='%1.0f%%', startangle=90, textprops={'fontsize': 9})
ax9.set_title('Suitability Distribution')

plt.tight_layout(rect=[0, 0, 1, 0.96])
_viz_path = os.path.join(RESULTS_DIR, f'FA4_climate_profile_{COUNTRY}.png')
plt.savefig(_viz_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'\u2705 Visualisation saved: {_viz_path}')

In [ ]:
# @title ### 8) Cache results to Shared Drive {"display-mode":"form"}

try:
    advance_notebook_progress(8)
except NameError:
    pass  # overall progress tracker not initialised (Config cell not yet run) -- safe to ignore

from google.colab import files
import json as _json

print(f'\u2601\ufe0f  Caching FA4 results to Shared Drive/{COUNTRY}...')

# --- dashboard patch 1: structured summary export ---
# Everything below was previously only printed to notebook output. Exporting it
# as JSON lets a downstream tool (e.g. the Streamlit/React dashboard) read the AEZ
# summary numbers without parsing an image or rerunning PyAEZ.
_summary = {
    'country':              COUNTRY,
    'period':               f'{LT_START}-{LT_END}',
    'precip_source':        PRECIP_SOURCE,
    'thermal_class':        int(thermal_class) if thermal_class is not None else None,
    'thermal_class_label':  THERMAL_CLASSES.get(thermal_class, 'Unknown'),
    'moisture_class':       int(moisture_class) if moisture_class is not None else None,
    'moisture_class_label': MOISTURE_CLASSES.get(moisture_class, 'Unknown'),
    'lgp_days_per_year':    float(lgp_value) if lgp_value is not None else None,
    'annual_tavg_c':        float(tavg_annual),
    'annual_precip_mm':     float(precip_annual),
    'annual_pet_mm':        float(pet_monthly_mm.sum()),
    'aridity_index':        float(precip_annual / pet_monthly_mm.sum()),
    'has_spatial_maps':     bool(HAS_SPATIAL_MAPS),
    'generated_date':       date.today().isoformat(),
}
_summary_path = os.path.join(RESULTS_DIR, f'FA4_summary_{COUNTRY}.json')
with open(_summary_path, 'w') as f:
    _json.dump(_summary, f, indent=2)
print(f'\u2705 Summary JSON written: {_summary_path}')

# --- dashboard patch 2: PRECIP_SOURCE baked into every cached filename ---
# so results from different sources (CHIRPS/TAMSAT/ERA5/IMERG) don't overwrite
# each other on the Shared Drive -- lets the dashboard offer a source switcher.
_outputs = [
    (f'FA4_summary_{COUNTRY}_{PRECIP_SOURCE}.json',
     _summary_path),
    (f'FA4_AEZ_{COUNTRY}_{PRECIP_SOURCE}.png',
     os.path.join(RESULTS_DIR, f'FA4_AEZ_{COUNTRY}.png')),
    (f'FA4_crop_suitability_{COUNTRY}_{PRECIP_SOURCE}.csv',
     os.path.join(RESULTS_DIR, f'FA4_crop_suitability_{COUNTRY}.csv')),
    (f'FA4_climate_profile_{COUNTRY}_{PRECIP_SOURCE}.png',
     os.path.join(RESULTS_DIR, f'FA4_climate_profile_{COUNTRY}.png')),
]

for drive_name, local_path in _outputs:
    if os.path.exists(local_path):
        cache_to_drive(local_path, drive_name)
    else:
        print(f'   \u26a0\ufe0f  {drive_name} not found (run earlier cells first)')

print('\n\U0001f4e5 Downloading results...')
for _, local_path in _outputs:
    if os.path.exists(local_path):
        files.download(local_path)

print('\n' + '\u2550' * 60)
print(f'  FOCUS AREA 4 COMPLETE \u2014 {COUNTRY}')
print('\u2550' * 60)
print(f'  Thermal class  : {thermal_class} \u2014 {THERMAL_CLASSES.get(thermal_class,"?")[:40]}')
print(f'  Moisture zone  : {moisture_class} \u2014 {MOISTURE_CLASSES.get(moisture_class,"?")[:40]}')
print(f'  LGP            : {lgp_value} days/year')
print()

_high = df_suitability[df_suitability['Suitability']=='High']['Crop'].tolist()
_mod  = df_suitability[df_suitability['Suitability']=='Moderate']['Crop'].tolist()
_marg = df_suitability[df_suitability['Suitability']=='Marginal']['Crop'].tolist()
_none = df_suitability[df_suitability['Suitability']=='Not suitable']['Crop'].tolist()

if _high: print(f'  \u2705 High suitability     : {", ".join(_high)}')
if _mod:  print(f'  \U0001f7e1 Moderate suitability : {", ".join(_mod)}')
if _marg: print(f'  \U0001f7e0 Marginal suitability : {", ".join(_marg)}')
if _none: print(f'  \u274c Not suitable         : {", ".join(_none)}')
print('\u2550' * 60)